# QAT vs PTQ mini-experiment (D13 step 2) — GRU
Compares fp32 baseline vs custom **QAT** vs **PTQ** on the same 200k/10k
search-regime slice, same optimizer/seed/epochs as the hyperparameter search.
Reports validation loss + BLEU (+ optional METEOR) and the int8 size estimate.


In [ ]:
import os, sys, json, glob, subprocess
from pathlib import Path
import torch

MODEL = 'transformer'
BLEU_SAMPLES = 1000     # val pairs to greedy-decode for BLEU/METEOR (0 = skip)
CALIB_BATCHES = 50      # PTQ calibration batches
USE_METEOR = True       # Indonesian-aware METEOR (needs nltk data + Sastrawi below)

DATASET_PATH = Path(glob.glob('/kaggle/input/**/train.tsv', recursive=True)[0]).parent
TOKENIZER_MODEL = Path(glob.glob('/kaggle/input/**/spm_en_id.model', recursive=True)[0])

REPO_URL = 'https://github.com/0wLzz/Edge-NMT.git'
if not os.path.isdir('Edge-NMT'):
    subprocess.run(['git','clone','--depth','1',REPO_URL], check=True)
    
os.chdir('/kaggle/working/Edge-NMT')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
print('dataset:', DATASET_PATH)

In [ ]:
# Dependencies (same core set as the search; + nltk/Sastrawi for METEOR)
pkgs = ['sentencepiece','sacrebleu','pyyaml','optuna','torchinfo']
if USE_METEOR:
    pkgs += ['nltk','Sastrawi']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=True)

if USE_METEOR:
    import nltk
    nltk.download('wordnet'); nltk.download('omw-1.4')

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), 'sm_%d%d' % cap)
else:
    print('No GPU detected, using CPU.')

In [ ]:
CFG = 'configs/config.yaml'

def run_module(mod, *args):
    print('RUN', mod, *args, flush=True)
    subprocess.run([sys.executable,'-m',mod,*map(str,args)], check=True)

exp_args = [
    '--arch', MODEL,
    '--config', CFG,
    '--dataset-dir', DATASET_PATH,
    '--tokenizer-model', TOKENIZER_MODEL,
    '--bleu-samples', BLEU_SAMPLES,
    '--calib-batches', CALIB_BATCHES,
    '--epochs', 1,
]

if USE_METEOR:
    exp_args.append('--meteor')

# Run Experiments
run_module('model.experiments.quant_ptq_vs_qat', *exp_args)

In [ ]:
# Show the result table
p = f'results/quant_experiment/{MODEL}_qat_vs_ptq.json'
if os.path.exists(p):
    r = json.load(open(p))
    print('config:', r['config']['epochs'], 'epochs | subset', r['config']['subset_size'],
          '| val', r['config']['val_size'], '| hparams', r['config']['hparams'])
    print()
    res = r['results']
    cols = ['val_loss','bleu'] + (['meteor'] if r['config']['meteor'] else []) + ['int8_size_mb','fp32_size_mb']
    print('variant'.ljust(10) + ''.join(c.rjust(14) for c in cols))
    for name in ('baseline','qat','ptq'):
        row = res[name]
        print(name.ljust(10) + ''.join(str(row.get(c,'')).rjust(14) for c in cols))
    print('\nDeltas vs fp32:')
    for name in ('qat','ptq'):
        row = res[name]
        print(' ', name, 'val_loss', row.get('val_loss_delta_vs_fp32'), '| BLEU', row.get('bleu_delta_vs_fp32'))
else:
    print('No results found at', p)